In [ ]:
"""
Garbage Detection Model Training Script
This script trains a CNN model to classify different types of garbage
"""

import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import json

# Configuration
IMG_SIZE = 224  # Image size for model input
BATCH_SIZE = 32
EPOCHS = 50
DATASET_PATH = "dataset/images"


CATEGORIES = [
    'plastic',
    'paper',
    'metal',
    'glass',
    'organic',
    'cardboard'
]

def create_dataset_structure():
   
    print("Creating dataset folder structure...")
    for category in CATEGORIES:
        os.makedirs(f"{DATASET_PATH}/{category}", exist_ok=True)
    
    for category in CATEGORIES:
        print(f"   - {DATASET_PATH}/{category}/")
    print("\nMove your captured images into the appropriate category folders before training.\n")

def load_and_preprocess_data():
    """
    Load images from organized folders and prepare for training
    """
    print("Loading dataset...")
    data = []
    labels = []
    
    for idx, category in enumerate(CATEGORIES):
        path = os.path.join(DATASET_PATH, category)
        if not os.path.exists(path):
            print(f"Warning: {path} does not exist. Skipping...")
            continue
        
        images = os.listdir(path)
        print(f"Loading {len(images)} images from {category}...")
        
        for img_name in images:
            try:
                img_path = os.path.join(path, img_name)
                img = cv2.imread(img_path)
                if img is None:
                    continue
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                data.append(img)
                labels.append(idx)
            except Exception as e:
                print(f"Error loading {img_name}: {e}")
    
    if len(data) == 0:
        raise ValueError("No images found! Please organize images into category folders.")
    
    data = np.array(data, dtype='float32') / 255.0  # Normalize
    labels = np.array(labels)
    
    print(f"\nTotal images loaded: {len(data)}")
    print(f"Image shape: {data[0].shape}")
    print(f"Number of classes: {len(CATEGORIES)}")
    
    return data, labels

def visualize_samples(data, labels, num_samples=10):
    """
    Visualize sample images from the dataset
    """
    plt.figure(figsize=(15, 6))
    for i in range(min(num_samples, len(data))):
        plt.subplot(2, 5, i + 1)
        plt.imshow(data[i])
        plt.title(CATEGORIES[labels[i]])
        plt.axis('off')
    plt.tight_layout()
    plt.savefig('dataset_samples.png')
    print("Sample images saved as 'dataset_samples.png'")
    plt.close()

def create_cnn_model(num_classes):
    """
    Create a CNN model for garbage classification
    Using MobileNetV2 as base for efficient inference
    """
    base_model = keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )
    
    # Freeze base model initially
    base_model.trainable = False
    
    # Create model
    model = keras.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    return model, base_model

def create_simple_cnn_model(num_classes):

    model = keras.Sequential([
        # First Convolutional Block
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Second Convolutional Block
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Third Convolutional Block
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Dense Layers
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    return model

def train_model(use_transfer_learning=True):
    """
    Main training function
    """
    # Create dataset structure
    create_dataset_structure()
    
    # Load data
    data, labels = load_and_preprocess_data()
    
    # Visualize samples
    visualize_samples(data, labels)
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        data, labels, test_size=0.2, random_state=42, stratify=labels
    )
    
    print(f"\nTraining samples: {len(X_train)}")
    print(f"Testing samples: {len(X_test)}")
    
    # Convert labels to categorical
    y_train_cat = keras.utils.to_categorical(y_train, len(CATEGORIES))
    y_test_cat = keras.utils.to_categorical(y_test, len(CATEGORIES))
    
    # Data augmentation
    datagen = ImageDataGenerator(
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        horizontal_flip=True,
        zoom_range=0.2,
        shear_range=0.2,
        fill_mode='nearest'
    )
    
    # Create model
    if use_transfer_learning and len(X_train) > 500:
        print("\nUsing Transfer Learning (MobileNetV2)...")
        model, base_model = create_cnn_model(len(CATEGORIES))
    else:
        print("\nUsing Simple CNN (better for small datasets)...")
        model = create_simple_cnn_model(len(CATEGORIES))
        base_model = None
    
    # Compile model
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    print("\nModel Summary:")
    model.summary()
    
    # Callbacks
    callbacks = [
        ModelCheckpoint(
            'best_garbage_model.keras',
            save_best_only=True,
            monitor='val_accuracy',
            mode='max',
            verbose=1
        ),
        EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-7,
            verbose=1
        )
    ]
    
    # Train model
    print("\nStarting training...")
    history = model.fit(
        datagen.flow(X_train, y_train_cat, batch_size=BATCH_SIZE),
        epochs=EPOCHS,
        validation_data=(X_test, y_test_cat),
        callbacks=callbacks,
        verbose=1
    )
    
    # Fine-tuning (if using transfer learning)
    if base_model is not None and len(X_train) > 1000:
        print("\n" + "="*50)
        print("Fine-tuning the model...")
        print("="*50)
        
        # Unfreeze base model
        base_model.trainable = True
        
        # Compile with lower learning rate
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=1e-5),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )
        
        # Continue training
        history_fine = model.fit(
            datagen.flow(X_train, y_train_cat, batch_size=BATCH_SIZE),
            epochs=20,
            validation_data=(X_test, y_test_cat),
            callbacks=callbacks,
            verbose=1
        )
    
    # Evaluate model
    print("\n" + "="*50)
    print("Evaluating model...")
    print("="*50)
    
    test_loss, test_acc = model.evaluate(X_test, y_test_cat, verbose=0)
    print(f"\nTest Accuracy: {test_acc*100:.2f}%")
    print(f"Test Loss: {test_loss:.4f}")
    
    # Predictions
    y_pred = model.predict(X_test)
    y_pred_classes = np.argmax(y_pred, axis=1)
    
    # Classification report
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred_classes, target_names=CATEGORIES))
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred_classes)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=CATEGORIES, yticklabels=CATEGORIES)
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig('confusion_matrix.png')
    print("Confusion matrix saved as 'confusion_matrix.png'")
    plt.close()
    
    # Plot training history
    plot_training_history(history)
    
    # Save model and metadata
    model.save('garbage_detector_model.keras')
    
    # Save model info
    model_info = {
        'categories': CATEGORIES,
        'img_size': IMG_SIZE,
        'test_accuracy': float(test_acc),
        'num_classes': len(CATEGORIES)
    }
    
    with open('model_info.json', 'w') as f:
        json.dump(model_info, f, indent=4)
    
    print("\n" + "="*50)
    print("Training completed!")
    print("="*50)
    print(f"Model saved as: garbage_detector_model.keras")
    print(f"Best model saved as: best_garbage_model.keras")
    print(f"Model info saved as: model_info.json")
    
    return model, history

def plot_training_history(history):
    """
    Plot training and validation accuracy/loss
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Accuracy
    ax1.plot(history.history['accuracy'], label='Training Accuracy')
    ax1.plot(history.history['val_accuracy'], label='Validation Accuracy')
    ax1.set_title('Model Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True)
    
    # Loss
    ax2.plot(history.history['loss'], label='Training Loss')
    ax2.plot(history.history['val_loss'], label='Validation Loss')
    ax2.set_title('Model Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.savefig('training_history.png')
    print("Training history saved as 'training_history.png'")
    plt.close()

if __name__ == "__main__":
    print("="*60)
    print("GARBAGE DETECTION MODEL TRAINING")
    print("="*60)
    
    # Check if organized dataset exists
    organized = all(os.path.exists(f"{DATASET_PATH}/{cat}") for cat in CATEGORIES)
    
    if not organized:
        print("\n Dataset not organized yet!")
        print("\nSteps to prepare your dataset:")
        print("1. Run this script to create folder structure")
        print("2. Organize your captured images into category folders")
        create_dataset_structure()
    else:
        # Train the model
        model, history = train_model(use_transfer_learning=True)